In [58]:
import json
import os
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool

In [59]:
from dotenv import load_dotenv
_ = load_dotenv()

import os 
API_KEY = os.getenv("OPENAI_API_KEY")

In [60]:
from crewai.llms.providers.openai.completion import OpenAICompletion
llm = OpenAICompletion(model="gpt-3.5-turbo-0125", api_key=API_KEY)


In [61]:
csv_carteira = CSVSearchTool(csv="ativos.csv")

Agente 1 - Gerente do cliente

In [62]:
gerente_cliente = Agent(
    role="Gerente de Carteira do Cliente",
    goal="Obtenha a pergunta do cliente sobre o ativo {ticket} e pesquise as ações no arquivo CSV da carteira do cliente",
    backstory="""
    Você é o gerente de clientes da carteira de investimentos do cliente.
    Você é o primeiro contato do cliente e fornece as informações para as demais análises com o ticket do ativo e informações de carteira necessárias
    """,
    verbose=True,
    max_iter=5,
    tools=[csv_carteira],
    memory=True
)

In [63]:
obter_carteira_cliente = Task(
    description=""",
    Use a pergunta do cliente e encontre o ativo {ticket} no arquivo CSV.
    Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o número total de ações em posse.
    """,
    expected_output="Se o cliente possuir os ativos, forneça o preço médio e o total de ações dos ativos",
    agent=gerente_cliente
)

In [64]:
import yfinance as yf

def pega_preco_ativo(ticket):
    data_final = datetime.today()
    data_inicial = data_final - timedelta(days=365)
    ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
    return ativo

In [65]:
resultado = pega_preco_ativo("PETR4.SA")
resultado

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA
Date,,,,,
2025-05-06,27.378172,27.596107,27.205639,27.341850,52751300
2025-05-07,27.505302,27.514382,27.151156,27.514382,35050400
2025-05-08,27.886688,28.177271,27.650593,27.777721,44665000
2025-05-09,28.068302,28.313480,27.823124,28.240835,25075600
2025-05-12,28.740271,29.212463,28.740271,28.967287,53493600
...,...,...,...,...,...
2026-04-28,47.520000,48.040001,47.459999,47.650002,29389700
2026-04-29,48.959999,49.299999,48.000000,48.099998,47685000


Agente 2 - Analista de Ações

In [66]:
analista_acoes = Agent(
    role="Analista senior de preço de ações",
    goal="Encontre o preço da ação {ticket} e analise suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago",
    backstory="""
    Você é um analista de  muito experiente.
    Você é responsável por analisar os ativos e fornecer informações sobre o preço do ativo para o gerente de  e fazer previsões sobre seu preço futuro.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [67]:
from crewai.tools import BaseTool
from pydantic import Field

In [68]:
class YahooFinanceTool(BaseTool):
    name: str = "Yahoo Finance Tool"
    description: str = "Use esta ferramenta para obter informações sobre o preço de um ativo, no último ano, usando a biblioteca yfinance. Forneça o ticket do ativo para obter as informações necessárias."

    def _run(self, ticket: str):
        """Executa a busca de preços de ações para o ativo fornecido usando a biblioteca yfinance."""
        try:
            data_final = datetime.today()
            data_inicial = data_final - timedelta(days=365)
            ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
            return ativo.to_dict()
        except Exception as e:
            return f"Erro ao obter dados do Yahoo Finance: {str(e)}"

In [69]:
yfinance_tool = YahooFinanceTool()
type(yfinance_tool)

__main__.YahooFinanceTool

In [70]:
obter_preco_acao = Task(
    description="""
    Use a ferramenta Yahoo Finance Tool para obter o preço da ação {ticket} e analisar suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago.
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago e especifique uma tendencia atual do preço da acao tanto para cima quanto para baixo",
    agent=analista_acoes,
    tools=[yfinance_tool]
)

Agente 3 - Analista de notícias

In [71]:
analista_noticias = Agent(
    role="Analista senior de notícias de ações",
    goal="Encontre as últimas notícias sobre o ativo {ticket} e analise seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira",
    backstory="""
    Você é um analista muito experiente.
    Você é responsável por analisar as notícias relacionadas aos ativos e fornecer informações sobre o impacto potencial dessas notícias no preço do ativo para o gerente de carteira.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [72]:
from langchain_community.tools import DuckDuckGoSearchResults
searchTool = DuckDuckGoSearchResults(backend="news", num_results=10)

In [73]:
obter_noticias = Task(
    description=f"""
    Use a ferramenta DuckDuckGo News Tool para obter as últimas notícias sobre o ativo e analisar seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira.
    A data atual é {datetime.now()}
    Componha os resultados em um relatório útil
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de analisar o impacto potencial das notícias no preço do ativo.",
    agent=analista_noticias,
    tool=[searchTool]
)

Agente 4 - Analista chefe de ações

In [74]:
analista_chefe = Agent(
    role="Analista chefe de investimentos",
    goal="Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.",
    backstory="""
    Você é um analista chefe de investimentos altamente experiente.
    Você é responsável por revisar as análises fornecidas pelos analistas de ações e notícias, e fornecer uma recomendação final para o gerente de carteira com base em todas as informações disponíveis.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [75]:
recomendar_acao = Task(
    description="""
    Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.
    Se os relatórios não forem conclusivos, pode solicitar mais análises ou informações adicionais para chegar a uma recomendação mais informada.
    """,
    expected_output="Forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo.",
    agent=analista_chefe,
    context=[obter_carteira_cliente, obter_preco_acao, obter_noticias]
)

Agente 5 - Redator

In [76]:
redator = Agent(
    role="Redator de Relatórios de Investimentos",
    goal="Com base na recomendação final do analista chefe, redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, incluindo as análises de preço e notícias.",
    backstory="""
    Você é um redator experiente especializado em relatórios de investimentos.
    Sua tarefa é transformar a recomendação técnica do analista chefe em um relatório compreensível e útil para o cliente, destacando os pontos-chave e explicando as razões por trás da recomendação.
    Escreva em uma liguagem acessível, evitando jargões técnicos, para garantir que o cliente possa entender claramente a recomendação e os fatores que a influenciaram.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [77]:
escrever_boletim = Task(
    description="""
    Com base na recomendação final do analista chefe, redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, incluindo as análises de preço e notícias.
    Escreva em uma linguagem acessível, evitando jargões técnicos, para garantir que o cliente possa entender claramente a recomendação e os fatores que a influenciaram.
    Escreva com mais ou menos 6 paragráfos, destacando os pontos-chave da análise e explicando as razões por trás da recomendação de forma clara e objetiva.
    """,
    expected_output="""
    Redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, 
    incluindo as análises de preço e notícias. Escreva formatado como markdown, com títulos e subtítulos para organizar as informações de 
    forma clara e fácil de ler.
    Deve conter uma introdução explicando o objetivo do relatório, uma seção de análise de preço destacando as tendências e comparações com o preço pago,
    uma seção de análise de notícias explicando o impacto potencial das notícias no preço do ativo, e uma conclusão com a recomendação final de 
    compra, venda ou manutenção, resumindo os principais pontos que levaram a essa recomendação.
    """,
    agent=redator,
    context=[obter_preco_acao, obter_noticias, recomendar_acao]
)

In [78]:
crew = Crew(
    agents=[gerente_cliente, analista_acoes, analista_noticias, analista_chefe, redator],
    tasks=[obter_carteira_cliente, obter_preco_acao, obter_noticias, recomendar_acao, escrever_boletim],
    verbose=True,
    process = Process.hierarchical,
    full_output=True,
    share_crew=False,
    max_iter=15,
    manager_llm=llm
)

In [79]:
result = crew.kickoff(inputs={"ticket": "De a sua opinião sobre o ativo PETR4.SA, considerando as análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a carteira do cliente."})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 69b02a13-6f17-452c-8329-4de0390ff26f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: ,                                                                                                        │
│      Use a pergunta do cliente e encontre o ativo De a sua opinião sobre o ativo PETR4.SA, considerando as      │
│  análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a        │
│  carteira do cliente. no arquivo CSV.                                                                           │
│      Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o       │
│  número total de ações em posse.                                                                                │
│                                                                                                                 │
│  ID: 7548b78d-8a17-4301-af42-1c5bde1b2a07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: ,                                                                                                        │
│      Use a pergunta do cliente e encontre o ativo De a sua opinião sobre o ativo PETR4.SA, considerando as      │
│  análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a        │
│  carteira do cliente. no arquivo CSV.                                                                           │
│      Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o       │
│  número total de ações em posse.                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_csvs_content                                                                                    │
│  Args: {'search_query': 'PETR4.SA'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_csvs_content executed with result: Relevant Content:
Headers: Código | Nome do Ativo | Setor | Preço Atual (R$) | Preço Médio (R$) | Total de Ações
--------------------------------------------------
Row 
Row 1: Código: ABEV3 | Nome do ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│      Use a ferramenta Yahoo Finance Tool para obter o preço da ação De a sua opinião sobre o ativo PETR4.SA,    │
│  considerando as análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou          │
│  manutenção para a carteira do cliente. e analisar suas tendências para fornecer uma recomendação de compra,    │
│  venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago.                     │
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 69b02a13-6f17-452c-8329-4de0390ff26f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

LLMContextLengthExceededError: LLM context length exceeded. Original error: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 29246 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
Consider using a smaller input or implementing a text splitting strategy.